# 02 — Feature Importance (optional)

Evaluates which photometric features (magnitudes and colors) are most
relevant for predicting each stellar parameter.

**This notebook is optional.** Run it if you want to train models using
only the most important features instead of all available features.

**Workflow:**
1. Load catalog and assemble feature DataFrame
2. Compute feature importance (impurity-based or permutation)
3. Save selected features to JSON — used by `03_tuning.ipynb`

**Output:** `features/<param>_features.json` for each stellar parameter

## Configuration

In [ ]:
import os
import json
import pandas as pd
import minas as mg
from minas.evaluation.feature_selection import (
    get_important_features,
    get_important_features_xgb,
    get_permutation_importance_rf,
    get_permutation_importance_xgb,
)

# ── Files ─────────────────────────────────────────────────────────────────────
INPUT_FILE   = 'input_catalog.csv'  # output from 01_data_creation.ipynb
INDEX_COL    = 'ID'                 # unique identifier column (or None)
OUTPUT_DIR   = 'features'           # folder where feature JSON files are saved

# ── Survey ────────────────────────────────────────────────────────────────────
SURVEY = 'SPLUS'   # 'SPLUS', 'JPLUS', 'JPAS', etc.

# ── Parameters to evaluate ───────────────────────────────────────────────────
PARAM_LIST = ['teff', 'logg', 'feh']

# ── Distance column (set to None if absolute magnitudes are not needed) ───────
DIST_COL = 'Dist'

# ── Feature importance settings ───────────────────────────────────────────────
MODEL_TYPE       = 'XGB'       # 'RF' or 'XGB'
METHOD           = 'impurity'  # 'impurity' (fast) or 'permutation' (more reliable)
N_FEATURES       = 20          # number of top features to select and save
SCORING          = 'r2'        # 'r2' or 'mad' (only used for permutation method)
N_ESTIMATORS     = 100
RANDOM_STATE     = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Survey  : {SURVEY}')
print(f'Method  : {METHOD} importance  |  Model: {MODEL_TYPE}')
print(f'Top-N   : {N_FEATURES} features per parameter')

## Step 1 — Load catalog and assemble feature DataFrame

In [ ]:
def get_column(df, aliases):
    """Return the first matching column from a list of aliases."""
    for col in aliases:
        if col in df.columns:
            return col
    raise KeyError(f'No column found for aliases: {aliases}')


df_base = pd.read_csv(INPUT_FILE)
if INDEX_COL and INDEX_COL in df_base.columns:
    df_base = df_base.set_index(INDEX_COL)

filters = mg.FILTERS[SURVEY]
print(f'Catalog loaded: {len(df_base):,} objects')

# Pre-process: compute work DataFrame for each parameter
preprocessed = {}
for param in PARAM_LIST:
    df = df_base.copy()

    # Remove invalid values for this parameter
    param_col = get_column(df, mg.PARAM_ALIASES[param])
    df = df[df[param_col] != -9999].dropna(subset=[param_col])

    # Compute absolute magnitudes if distance column is available
    if DIST_COL and DIST_COL in df.columns:
        df = mg.preprocess.calculate_abs_mag(df, filters, DIST_COL)

    # Assemble feature DataFrame
    work_df = mg.preprocess.assemble_work_df(
        df=df,
        filters=filters,
        correction_pairs=None,
        add_colors=True,
        verbose=False,
    )

    preprocessed[param] = {'df': df, 'work_df': work_df, 'param_col': param_col}
    print(f'  [{param:4s}] {len(df):,} objects  |  {work_df.shape[1]} features')

## Step 2 — Compute feature importance

In [ ]:
# Select the importance function based on method and model type
_importance_fn = {
    ('impurity',    'RF') : get_important_features,
    ('impurity',    'XGB'): get_important_features_xgb,
    ('permutation', 'RF') : get_permutation_importance_rf,
    ('permutation', 'XGB'): get_permutation_importance_xgb,
}

importance_fn = _importance_fn.get((METHOD, MODEL_TYPE))
if importance_fn is None:
    raise ValueError(f"Invalid combination: METHOD='{METHOD}', MODEL_TYPE='{MODEL_TYPE}'")

results = {}

for param in PARAM_LIST:
    work_df   = preprocessed[param]['work_df']
    df        = preprocessed[param]['df']
    param_col = preprocessed[param]['param_col']

    print(f'Computing importance for [{param}]...')

    kwargs = dict(
        X=work_df,
        y=df[param_col],
        n_features_to_save=N_FEATURES,
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
    )
    if METHOD == 'permutation':
        kwargs['scoring'] = SCORING

    selected_features, df_feat = importance_fn(**kwargs)
    results[param] = {'selected_features': selected_features, 'df_feat': df_feat}

    print(f'  Top {N_FEATURES} features: {selected_features[:5]} ...')

print('\nDone.')

## Step 3 — Save selected features to JSON

In [ ]:
for param in PARAM_LIST:
    features = results[param]['selected_features']
    out_path = os.path.join(OUTPUT_DIR, f'{param}_features.json')
    with open(out_path, 'w') as f:
        json.dump(features, f, indent=2)
    print(f'Saved: {out_path}  ({len(features)} features)')

## Step 4 — Plot feature importance (optional)

In [ ]:
from minas.evaluation._graphics import plot_feature_importance

N_PLOT = 15   # number of features to show in the plot

for param in PARAM_LIST:
    df_feat = results[param]['df_feat']
    plot_feature_importance(df_feat, param=param, n_top_features=N_PLOT)